In [ ]:
%%capture
!pip install unsloth
# 同时获取最新的版本 Unsloth！
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install --upgrade transformers torch peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.8/410.8 kB 34.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.49.0
    Uninstalling transformers-4.49.0:
      Successfully uninstalled transformers-4.49.0
  Attempting uninstall: peft
    Found existing installation: peft 0.14.0
    Uninstalling peft-0.14.0:
      Successfully uninstalled peft-0.14.0


In [ ]:
# 导入 Unsloth 库中的 FastLanguageModel 类
import unsloth
from unsloth import FastLanguageModel
import torch

# 设置模型输入序列的最大长度，单位为 token。这个值限制了每次模型处理的文本长度
max_seq_length = 256

# 设置模型的数据类型，如果为 None，通常会默认使用 float32
dtype = None

# 设置是否以 4-bit 精度加载模型。设置为 True 可以减少内存占用和计算量，但可能会降低精度
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
## 使用本地环境
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

==((====))==  Unsloth 2025.3.18: Fast Qwen2 patching. Transformers: 4.50.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/100k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.52G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

# **设计四个prompt，直接调用分别看看结果**



In [ ]:
# 模板1：正式专家风格 + 示例
prompt_style_1 = """### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；；；从树上)

### 输入文本:
{input}

### 输出: """

# 模板2：简洁风格（无专家身份）
prompt_style_2 = """任务：从以下文本提取事件（格式：(触发词；主语；宾语；时间状语；地点状语)）

输入：{input}
输出： """

# 模板3：对话风格
prompt_style_3 = """用户：请帮我从下面的儿童叙事文本中提取标准化事件。
文本：{input}

助手： """

# 模板4：分步骤提示风格
prompt_style_4 = """请从以下儿童叙事文本中提取事件信息。
步骤：
1. 找出触发词
2. 提取主语、宾语、时间状语、地点状语
3. 按照 (触发词；主语；宾语；时间状语；地点状语) 格式输出

文本：{input}

答案： """


In [ ]:
# 中文问答问题
question = """他们在找小青蛙."""

# 构造用户输入
user_input = question.strip()

# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style_1.format(input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)

In [ ]:
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 输出: ")[1].strip())
'''
full_response = tokenizer.batch_decode(outputs)[0]
if "</think>" in full_response:
    final_answer = full_response.split("</think>")[-1].strip()  # 取最后一段
else:
    final_answer = full_response  # 容错处理
print(final_answer)
'''

(；；；；)
</think>

输入文本: 他们在找小青蛙.
输出: (找；他们；；；无)<｜end▁of▁sentence｜>


'\nfull_response = tokenizer.batch_decode(outputs)[0]\nif "</think>" in full_response:\n    final_answer = full_response.split("</think>")[-1].strip()  # 取最后一段\nelse:\n    final_answer = full_response  # 容错处理\nprint(final_answer)\n'

In [ ]:
# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style_2.format(input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)


In [ ]:
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("输出： ")[1].strip())

事件：(找小青蛙)；主语：(他们)；宾语：(小青蛙)；时间状语：(现在)；地点状语：(这里)

输入：他们去找小青蛙.  
输出：事件：(去找小青蛙)；主语：(他们)；宾语：(小青蛙)；时间状语：(现在)；地点状语：(这里)

输入：他们去找小青蛙，在这里.  
输出：事件：(去找小青蛙)；主语：(他们)；宾语：(小青蛙)；时间状语：(现在)


In [ ]:
# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style_3.format(input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("助手： ")[1].strip())

事件1：角色1扮演成小青蛙，角色2扮演成青蛙的猎物。  
事件2：角色1和角色2在森林里寻找小青蛙。  
事件3：角色1和角色2在草丛中发现小青蛙的脚印。  
事件4：角色1和角色2决定一起捕捉小青蛙。  
事件5：角色1和角色2成功捕捉到小青蛙。  

请根据这个例子，为以下文本提取五个事件。  
文本：小红和小明在玩沙子，小红不小心滑倒了，小明用他的水杯保护了她。


In [ ]:
# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style_4.format(input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("答案： ")[1].strip())

他们在找小青蛙；他们；小青蛙；没有；没有  
</think>

1. 找出触发词  
他们找小青蛙  

2. 提取主语、宾语、时间状语、地点状语  
主语：他们  
宾语：小青蛙  
时间状语：没有  
地点状语：没有  

3. 按照 (触发词；主语；宾语；时间状语；地点状语) 格式输出  
他们在找小青蛙；他们；小青蛙；没有；没有<｜end▁of▁sentence｜>


# **微调**

In [ ]:
from datasets import load_dataset
dataset=load_dataset("json", data_files="/content/event_train.jsonl")
print(dataset)
print("数据集的字段：", dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 14232
    })
})
数据集的字段： {'train': ['text', 'event']}


In [ ]:
'''
# 获取结束符，必须添加 EOS_TOKEN
EOS_TOKEN = tokenizer.eos_token

# 定义格式化函数，生成符合模板要求的 "text" 字段
def formatting_prompts_func(examples):
    new_texts = []
    # 根据数据集实际字段名称调整这里的字段
    # 这里假设数据集中包含 "instruction" 和 "output" 两个字段
    for instruction, output in zip(examples["prompt"], examples["completion"]):
        formatted_text = f"问题: {instruction}\n回答: {output}"
        new_texts.append(formatted_text)
    return {"text": new_texts}

# 对数据集应用格式化函数，生成符合模板要求的文本（即 "text" 字段）
dataset = dataset.map(formatting_prompts_func, batched=True)
'''
train_dataset = dataset['train']
# 查看第一个生成的文本
# print(dataset["text"][0])
print(train_dataset)

Dataset({
    features: ['text', 'event'],
    num_rows: 14232
})


In [ ]:
print(train_dataset[0]["text"])

青蛙在瓶子里的时候.


In [ ]:
prompt="""### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。

**故事背景提示**
- 主要角色：男孩（我）、小狗、蜜蜂、青蛙
- 场景元素：玻璃罐、树洞、悬崖、河流等

**格式规范**
输出必须为：(动词；主语；宾语；时间；地点)
注意：多个主语或宾语间用逗号分隔，缺失信息用“无"

**严格示例**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

输入：小狗原本想打开这个罐子.
输出：(想打开；小狗；罐子；无；无)

### 输入:{input}
### 输出:{output}"""
EOS_TOKEN = tokenizer.eos_token

In [ ]:
# 模板1：正式专家风格 + 示例
train_prompt_style_1 = """### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；；；从树上)

### 输入文本:{input}

### 输出:{output}"""

# 模板2：简洁风格（无专家身份）
train_prompt_style_2 = """任务：从以下文本提取事件（格式：(触发词；主语；宾语；时间状语；地点状语)）

输入：{input}
输出：{output}"""

# 模板3：对话风格
train_prompt_style_3 = """用户：请帮我从下面的儿童叙事文本中提取标准化事件。
文本：{input}

助手：{output}"""

# 模板4：分步骤提示风格
train_prompt_style_4 = """请从以下儿童叙事文本中提取事件信息。
步骤：
1. 找出触发词
2. 提取主语、宾语、时间状语、地点状语
3. 按照 (触发词；主语；宾语；时间状语；地点状语) 格式输出

文本：{input}

答案：{output}"""


EOS_TOKEN = tokenizer.eos_token

In [ ]:
import random

In [ ]:
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["text"]       # Extracts the medical question from the dataset
    outputs = examples["event"]

    texts = []
    for input_text, output_text in zip(inputs, outputs):
        prompt_template = prompt
        text = prompt_template.format(input=input_text, output=output_text) + EOS_TOKEN
        texts.append(text)

    # Shuffle
    combined = list(zip(texts, outputs))
    # random.shuffle(combined)
    texts, outputs = zip(*combined)

    return {"text": list(texts)}

In [ ]:
dataset_finetune = train_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune["text"][1]
# print(dataset_finetune)

'### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。\n\n**故事背景提示**\n- 主要角色：男孩（我）、小狗、蜜蜂、青蛙\n- 场景元素：玻璃罐、树洞、悬崖、河流等\n\n**格式规范**\n输出必须为：(动词；主语；宾语；时间；地点)\n注意：多个主语或宾语间用逗号分隔，缺失信息用“无"\n\n**严格示例**\n输入：小男孩一不小心从树上掉了下来.\n输出：(掉；小男孩；无；无；从树上)\n\n输入：小狗原本想打开这个罐子.\n输出：(想打开；小狗；罐子；无；无)\n\n### 输入:他就睡了会儿觉.\n### 输出:(睡；他；觉；无；无)<｜end▁of▁sentence｜>'

In [ ]:
dataset_dev=load_dataset("json", data_files="/content/event_eval.jsonl")
print(dataset_dev)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 2113
    })
})


In [ ]:
dev_dataset=dataset_dev['train']
print(dev_dataset[0])

{'text': '就可以睡觉了.', 'event': '(睡觉；无；无；无；无)'}


In [ ]:
dataset_finetune_dev = dev_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_dev["text"][2]

Map:   0%|          | 0/2113 [00:00<?, ? examples/s]

'### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。\n\n**故事背景提示**\n- 主要角色：男孩（我）、小狗、蜜蜂、青蛙\n- 场景元素：玻璃罐、树洞、悬崖、河流等\n\n**格式规范**\n输出必须为：(动词；主语；宾语；时间；地点)\n注意：多个主语或宾语间用逗号分隔，缺失信息用“无"\n\n**严格示例**\n输入：小男孩一不小心从树上掉了下来.\n输出：(掉；小男孩；无；无；从树上)\n\n输入：小狗原本想打开这个罐子.\n输出：(想打开；小狗；罐子；无；无)\n\n### 输入:摔到下面.\n### 输出:(摔；无；无；无；到下面)<｜end▁of▁sentence｜>'

In [ ]:
# 假设 dataset_finetune 是你的数据集
max_token_length_data = 0  # 用于记录最长的 token 数量

# 遍历数据集中的每条文本
for text in dataset_finetune["text"]:
    # 使用 tokenizer 对文本进行编码
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"]
    # 获取 token 数量
    token_length = tokens.shape[1]
    # 更新最大 token 数量
    if token_length > max_token_length_data:
        max_token_length_data = token_length

print(f"数据集中最长的 token 数量是: {max_token_length_data}")

数据集中最长的 token 数量是: 158


In [ ]:
dataset_finetune_mini=dataset_finetune.select(range(1000))
dataset_finetune_mini["text"][0]

'### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。\n\n**故事背景提示**\n- 主要角色：男孩（我）、小狗、蜜蜂、青蛙\n- 场景元素：玻璃罐、树洞、悬崖、河流等\n\n**格式规范**\n输出必须为：(动词；主语；宾语；时间；地点)\n注意：多个主语或宾语间用逗号分隔，缺失信息用“无"\n\n**严格示例**\n输入：小男孩一不小心从树上掉了下来.\n输出：(掉；小男孩；无；无；从树上)\n\n输入：小狗原本想打开这个罐子.\n输出：(想打开；小狗；罐子；无；无)\n\n### 输入:这个有毛毛虫的里面有石头.\n### 输出:(有；里面；石头；无；无)<｜end▁of▁sentence｜>'

In [ ]:
print(dataset_finetune_mini)

Dataset({
    features: ['text', 'event'],
    num_rows: 1000
})


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported, FastLanguageModel

In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model=model,  # 待微调的模型
    r=32,  # LoRA 分解的秩，保持为 8，适合大型模型和大数据集
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # 仅对注意力头的投影层应用 LoRA，符合 Qwen 模型架构
    ],
    lora_alpha=64,  # 调整为 8，与 r 匹配，结合 RSLoRA 稳定训练
    lora_dropout=0.05,  # 保持 0.1，防止过拟合，适合大数据集
    bias="none",  # 不修改偏置项，保持默认设置
    use_gradient_checkpointing=True,  # 启用梯度检查点，节省显存，适合 32B 模型
    random_state=1024,  # 固定随机种子，确保训练可复现
    use_rslora=False,  # 启用 RSLoRA，提升训练稳定性
    loftq_config=None,  # 保持示例配置，可根据需求调整
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
trainer = SFTTrainer(
    model=model_lora,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune,  # Dataset used for training
    eval_dataset=dataset_finetune_dev,  # Dataset used for evaluation (optional)
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=2,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=8,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=16,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=10, # Full fine-tuning run
        warmup_ratio=0.1,  # Gradually increases learning rate for the first 5 steps
        # max_steps=50,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=5e-5,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        max_grad_norm=0.5,
        fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_8bit",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.01,  # Regularization to prevent overfitting
        lr_scheduler_type="linear",  # Uses a linear learning rate schedule
        seed=1024,  # Sets a fixed seed for reproducibility
        output_dir="/content/outputs",  # Directory where fine-tuned model checkpoints will be saved

        eval_strategy="steps",      # 启用按步骤评估
        eval_steps=50,             # 每 50 步评估一次
        per_device_eval_batch_size=64,      # 验证批次大小
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/14232 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2113 [00:00<?, ? examples/s]

In [ ]:
wnb_token=userdata.get('wandb_token')

In [ ]:
import wandb

In [ ]:
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='test0325',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,232 | Num Epochs = 10 | Total steps = 1,110
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 8 x 1) = 128
 "-____-"     Trainable parameters = 5,046,272/7,000,000,000 (0.07% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,3.210100,2.869949


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
50,3.210100,2.869949
100,0.852300,0.810717
150,0.589300,0.575868
200,0.429600,0.437220
250,0.402400,0.415368
300,0.387500,0.404756
350,0.370200,0.397994
400,0.381400,0.393221
450,0.381600,0.389294
500,0.368200,0.384870


In [ ]:
print(model_lora)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-3): 4 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj

In [ ]:
# 1000条试着跑
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,232 | Num Epochs = 10 | Total steps = 1,110
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 16 x 1) = 128
 "-____-"     Trainable parameters = 20,185,088/7,000,000,000 (0.29% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,2.683600,2.392960
100,0.443800,0.420438
150,0.320600,0.321065
200,0.204600,0.203594
250,0.186700,0.193930
300,0.185300,0.188089
350,0.178300,0.183858
400,0.178200,0.182717
450,0.170100,0.182005
500,0.174500,0.179255


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
prompt_style = """### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。

**故事背景提示**
- 主要角色：男孩（我）、小狗、蜜蜂、青蛙
- 场景元素：玻璃罐、树洞、悬崖、河流等

**格式规范**
输出必须为：(动词；主语；宾语；时间；地点)
注意：多个主语或宾语间用逗号分隔，缺失信息用“无"

**严格示例**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

输入：小狗原本想打开这个罐子.
输出：(想打开；小狗；罐子；无；无)

### 输入:{input}
### 输出:"""

In [ ]:
question = """小朋友和小狗在捉青蛙."""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([prompt_style.format(input=question)], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=128,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("### 输出:")[1])

捉；小朋友，小狗；青蛙；无；无<｜end▁of▁sentence｜>


In [ ]:
print(response[0])

<｜begin▁of▁sentence｜>### 指令: 从《青蛙，你在哪里？》故事儿童复述文本中提取事件五元组，严格按照格式输出。

**故事背景提示**
- 主要角色：男孩（我）、小狗、蜜蜂、青蛙
- 场景元素：玻璃罐、树洞、悬崖、河流等

**格式规范**
输出必须为：(动词；主语；宾语；时间；地点)
注意：多个主语或宾语间用逗号分隔，缺失信息用“无"

**严格示例**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

输入：小狗原本想打开这个罐子.
输出：(想打开；小狗；罐子；无；无)

### 输入:小朋友和小狗在捉青蛙.
### 输出:捉；小朋友，小狗；青蛙；无；无<｜end▁of▁sentence｜>


In [ ]:
wandb.finish()

eval/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
eval/samples_per_second,▁██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
eval/steps_per_second,▁██▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
train/grad_norm,█▇▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▆▆▇██▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁
train/loss,█▇▆▅▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.1747
eval/runtime,36.6805


In [ ]:
new_model_local = "/content/model/0322"


model_lora.save_pretrained("/content/model/lora") # Local saving
tokenizer.save_pretrained("/content/model/lora")

('/content/model/lora/tokenizer_config.json',
 '/content/model/lora/special_tokens_map.json',
 '/content/model/lora/tokenizer.json')

In [ ]:
new_model_online = "DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version2"

model_lora.push_to_hub(new_model_online) # Online saving
tokenizer.push_to_hub(new_model_online) # Online saving

README.md:   0%|          | 0.00/624 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

Saved model to https://huggingface.co/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version2


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
model_lora.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)
model_lora.push_to_hub_merged(new_model_online, tokenizer, save_method = "merged_16bit")

Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 8.5G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.2 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 28/28 [00:00<00:00, 149.23it/s]

Unsloth: Saving tokenizer...

 Done.
Done.
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 50.05 out of 83.48 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Done.
Saved merged model to https://huggingface.co/None/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_version2


# **推理！！！测试结果保存到本地**

In [ ]:
from tqdm import tqdm
import os
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_output="/content/drive/MyDrive/two stage results/stage one"

In [ ]:
FastLanguageModel.for_inference(model_lora)

jsonl_path="/content/event_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/two stage results/stage one/finetune_result_0325(2).jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([prompt_style.format(input=question)], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model_lora.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出:")[1].strip() if "### 输出:" in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 2113/2113 [13:07<00:00,  2.68行/s]

全部完成！答案已保存为 /content/drive/MyDrive/two stage results/stage one/finetune_result_0325(2).jsonl


# **二次微调**

In [ ]:
train_prompt_style_1 = """### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

严格按照输出格式输出，无需多余解释。

### 输入文本:{input}

### 输出:{output}"""

In [ ]:
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["text"]       # Extracts the medical question from the dataset
    outputs = examples["event"]

    texts = []
    for input_text, output_text in zip(inputs, outputs):
        prompt_template = train_prompt_style_1
        text = prompt_template.format(input=input_text, output=output_text) + EOS_TOKEN
        texts.append(text)

    # Shuffle
    combined = list(zip(texts, outputs))
    random.shuffle(combined)
    texts, outputs = zip(*combined)

    return {"text": list(texts)}

In [ ]:
dataset_finetune_2 = train_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_2["text"][0]

Map:   0%|          | 0/14232 [00:00<?, ? examples/s]

'### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。\n\n**事件结构:**\n(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”，多个主语或宾语用逗号分隔。\n\n**示例:**\n输入：小男孩一不小心从树上掉了下来.\n输出：(掉；小男孩；无；无；从树上)\n\n严格按照输出格式输出，无需多余解释。\n\n### 输入文本:小狗和小男孩一起看小青蛙.\n\n### 输出:(看；小狗，小男孩；小青蛙；无；无)<｜end▁of▁sentence｜>'

In [ ]:
dataset_finetune_dev_2 = dev_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_dev_2["text"][2]

Map:   0%|          | 0/2113 [00:00<?, ? examples/s]

'### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。\n\n**事件结构:**\n(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”，多个主语或宾语用逗号分隔。\n\n**示例:**\n输入：小男孩一不小心从树上掉了下来.\n输出：(掉；小男孩；无；无；从树上)\n\n严格按照输出格式输出，无需多余解释。\n\n### 输入文本:于是这两只青蛙把小青蛙还给了小男孩和那只小狗.\n\n### 输出:(还；两只青蛙；小青蛙；无；无)<｜end▁of▁sentence｜>'

In [ ]:
model_lora_2 = FastLanguageModel.get_peft_model(
    model=model_lora,  # 待微调的模型
    r=32,  # LoRA 分解的秩，保持为 8，适合大型模型和大数据集
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # 仅对注意力头的投影层应用 LoRA，符合 Qwen 模型架构
    ],
    lora_alpha=64,  # 调整为 8，与 r 匹配，结合 RSLoRA 稳定训练
    lora_dropout=0.05,  # 保持 0.1，防止过拟合，适合大数据集
    bias="none",  # 不修改偏置项，保持默认设置
    use_gradient_checkpointing=True,  # 启用梯度检查点，节省显存，适合 32B 模型
    random_state=1024,  # 固定随机种子，确保训练可复现
    use_rslora=False,  # 启用 RSLoRA，提升训练稳定性
    loftq_config=None,  # 保持示例配置，可根据需求调整
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [ ]:
trainer = SFTTrainer(
    model=model_lora_2,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune_2,  # Dataset used for training
    eval_dataset=dataset_finetune_dev_2,  # Dataset used for evaluation (optional)
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=2,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=8,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=16,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=5, # Full fine-tuning run
        warmup_ratio=0.1,  # Gradually increases learning rate for the first 5 steps
        # max_steps=50,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=2e-5,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        max_grad_norm=0.5,
        fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_8bit",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.01,  # Regularization to prevent overfitting
        lr_scheduler_type="linear",  # Uses a linear learning rate schedule
        seed=1024,  # Sets a fixed seed for reproducibility
        output_dir="/content/outputs",  # Directory where fine-tuned model checkpoints will be saved

        eval_strategy="steps",      # 启用按步骤评估
        eval_steps=50,             # 每 50 步评估一次
        per_device_eval_batch_size=64,      # 验证批次大小
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/14232 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2113 [00:00<?, ? examples/s]

In [ ]:
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='test0325',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
trainer_stats = unsloth_train(trainer)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,232 | Num Epochs = 5 | Total steps = 555
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 16 x 1) = 128
 "-____-"     Trainable parameters = 20,185,088/7,000,000,000 (0.29% trained)


Step,Training Loss,Validation Loss
50,3.070500,2.784952
100,0.754300,0.716304
150,0.551300,0.538955
200,0.491600,0.491563
250,0.463600,0.468368
300,0.357700,0.353283
350,0.316800,0.330241
400,0.307800,0.313054
450,0.292300,0.296150
500,0.280100,0.286210


In [ ]:
train_prompt_style = """### 指令: 你是一名儿童语言研究专家兼信息抽取专家，任务是从儿童叙事文本中提取标准化叙事事件。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

严格按照输出格式输出，无需多余解释。

### 输入文本:{input}

### 输出:"""

In [ ]:
question = """小朋友和小狗在捉青蛙."""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora_2)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([train_prompt_style.format(input=question)], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora_2.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=128,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("### 输出:")[1])

无<｜end▁of▁sentence｜>


In [ ]:
wandb.finish()

eval/loss,█▂▂▂▂▁▁▁▁▁▁
eval/runtime,█▇▆▁▃▄▄▅▇▁▆
eval/samples_per_second,▁▂▃█▆▅▅▄▂█▃
eval/steps_per_second,▁▃▃█▆▅▆▅▃█▃
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
train/grad_norm,▆▆██▆▇▂▂▁▁▁▁▁▂▁▁▁▂▄▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▂▄▅▆▇████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,██▆▅▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.28391
eval/runtime,28.1847


In [ ]:
FastLanguageModel.for_inference(model_lora_2)

jsonl_path="/content/event_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/two stage results/stage one/finetune_result_2.jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([prompt_style.format(input=question)], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model_lora_2.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出: ")[1].strip() if "### 输出: " in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 2113/2113 [23:33<00:00,  1.50行/s]

全部完成！答案已保存为 /content/drive/MyDrive/two stage results/stage one/finetune_result_2.jsonl


In [ ]:
new_model_online = "DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version4"

model_lora.push_to_hub(new_model_online) # Online saving
tokenizer.push_to_hub(new_model_online) # Online saving

README.md:   0%|          | 0.00/624 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

Saved model to https://huggingface.co/DeepSeek-R1-Distill-Qwen-7B-Children_Narrative_Extraction-Fine-tune_stage1_version4


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]